# 📡 Telecom X — Análisis de Evasión de Clientes (Churn)

> **Challenge Data Science | ONE G9 — Alura LATAM**

---

## 🎯 Objetivo

Analizar el comportamiento de los clientes de **Telecom X** para identificar patrones asociados a la **evasión de clientes (churn)**, es decir, clientes que cancelan sus servicios. El análisis cubre:

1. Carga y extracción de datos desde API
2. Exploración de la estructura del dataset
3. Verificación de calidad de datos
4. Limpieza y tratamiento de inconsistencias
5. Análisis descriptivo estadístico
6. Distribución del Churn
7. Churn por variables categóricas
8. Churn por variables numéricas

---

**Repositorio:** [GitHub - Challenge Telecom X Part 01](https://github.com/AlanSebastianArce/Challenge-Telecom-X-Part-01-G9-ONE.git)


## 📦 Instalación e Importación de Librerías

In [ ]:
# ─── Instalaciones (solo necesario en Colab) ─────────────────────────────────
# !pip install plotly --quiet

# ─── Librerías estándar ───────────────────────────────────────────────────────
import json
import warnings
warnings.filterwarnings('ignore')

# ─── Manipulación y análisis de datos ─────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Visualización estática ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ─── Visualización interactiva ─────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── Solicitudes HTTP ──────────────────────────────────────────────────────────
import requests

# ─── Configuración global de visualización ────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# Paleta personalizada para churn
CHURN_PALETTE = {'No': '#4C72B0', 'Yes': '#DD8452'}

print('✅ Librerías cargadas correctamente.')

---
## 1️⃣ Carga de Datos desde la API

Los datos se encuentran publicados en GitHub en formato **JSON**. Los cargamos directamente con `requests` y los normalizamos a un `DataFrame` de pandas.

**Fuente:** `https://raw.githubusercontent.com/AlanSebastianArce/Challenge-Telecom-X-Part-01-G9-ONE/refs/heads/main/telecom-x-database/Telecom-X-Data.json`

In [ ]:
# ─── URL de la API (raw GitHub) ───────────────────────────────────────────────
URL = (
    'https://raw.githubusercontent.com/AlanSebastianArce/'
    'Challenge-Telecom-X-Part-01-G9-ONE/refs/heads/main/'
    'telecom-x-database/Telecom-X-Data.json'
)

# ─── Solicitud HTTP ───────────────────────────────────────────────────────────
response = requests.get(URL)
response.raise_for_status()   # lanza error si el código HTTP no es 200

# ─── Parseo del JSON ──────────────────────────────────────────────────────────
raw_data = response.json()

print(f'📥 Tipo de objeto recibido: {type(raw_data)}')
if isinstance(raw_data, list):
    print(f'   Número de registros en lista: {len(raw_data)}')
elif isinstance(raw_data, dict):
    print(f'   Claves de primer nivel: {list(raw_data.keys())}')

In [ ]:
# ─── Normalización a DataFrame (maneja JSON anidado) ─────────────────────────
df = pd.json_normalize(raw_data)

print(f'✅ DataFrame creado exitosamente.')
print(f'   Filas   : {df.shape[0]:,}')
print(f'   Columnas: {df.shape[1]}')
print()
df.head(3)

---
## 2️⃣ Exploración de la Estructura del Dataset

Antes de limpiar o analizar, necesitamos entender **qué contiene** el dataset: tipos de datos, primeros registros y un vistazo al diccionario de variables.

### 📖 Diccionario de Datos (resumen)

| Columna | Descripción | Tipo |
|---|---|---|
| `customerID` | Identificador único del cliente | String |
| `Churn` | ¿El cliente canceló? (Yes / No) | Categórica |
| `gender` | Género del cliente | Categórica |
| `SeniorCitizen` | ¿Es adulto mayor? (1 / 0) | Binaria |
| `Partner` | ¿Tiene pareja? (Yes / No) | Categórica |
| `Dependents` | ¿Tiene dependientes? (Yes / No) | Categórica |
| `tenure` | Meses como cliente | Numérica |
| `PhoneService` | ¿Tiene servicio telefónico? | Categórica |
| `MultipleLines` | ¿Tiene múltiples líneas? | Categórica |
| `InternetService` | Tipo de servicio de internet | Categórica |
| `OnlineSecurity` | ¿Tiene seguridad en línea? | Categórica |
| `OnlineBackup` | ¿Tiene respaldo en línea? | Categórica |
| `DeviceProtection` | ¿Tiene protección de dispositivo? | Categórica |
| `TechSupport` | ¿Tiene soporte técnico? | Categórica |
| `StreamingTV` | ¿Tiene streaming de TV? | Categórica |
| `StreamingMovies` | ¿Tiene streaming de películas? | Categórica |
| `Contract` | Tipo de contrato | Categórica |
| `PaperlessBilling` | ¿Facturación sin papel? | Categórica |
| `PaymentMethod` | Método de pago | Categórica |
| `MonthlyCharges` | Cargo mensual en USD | Numérica |
| `TotalCharges` | Total pagado en USD | Numérica |


In [ ]:
# ─── Tipos de datos y valores no nulos ───────────────────────────────────────
print('📋 Información general del DataFrame:')
print('=' * 50)
df.info()

In [ ]:
# ─── Primeros registros ───────────────────────────────────────────────────────
print('🔍 Primeros 5 registros:')
df.head()

In [ ]:
# ─── Columnas del dataset ─────────────────────────────────────────────────────
print(f'📌 Total de columnas: {len(df.columns)}\n')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

In [ ]:
# ─── Valores únicos por columna categórica ────────────────────────────────────
cat_cols = df.select_dtypes(include='object').columns.tolist()

print('🔑 Valores únicos por columna categórica:\n')
for col in cat_cols:
    unique_vals = df[col].unique()
    print(f'  [{col}] → {unique_vals}')

---
## 3️⃣ Verificación de Calidad de Datos

Buscamos:
- ❌ Valores nulos o faltantes
- 🔁 Registros duplicados
- ⚠️ Errores de formato (espacios, capitalización)
- 🔄 Inconsistencias en categorías


In [ ]:
# ─── Valores nulos ────────────────────────────────────────────────────────────
null_summary = pd.DataFrame({
    'Nulos':       df.isnull().sum(),
    'Porcentaje':  (df.isnull().sum() / len(df) * 100).round(2)
})
null_summary = null_summary[null_summary['Nulos'] > 0].sort_values('Nulos', ascending=False)

if null_summary.empty:
    print('✅ No se detectaron valores nulos explícitos (NaN).')
else:
    print('⚠️  Columnas con valores nulos:')
    display(null_summary)

In [ ]:
# ─── Duplicados ───────────────────────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f'🔁 Registros duplicados: {n_dup}')

# Chequeo de IDs duplicados si existe la columna
if 'customerID' in df.columns:
    n_id_dup = df['customerID'].duplicated().sum()
    print(f'🔁 IDs de cliente duplicados: {n_id_dup}')

In [ ]:
# ─── Valores en blanco / cadenas vacías ──────────────────────────────────────
print('🔍 Revisando strings vacíos o con solo espacios en columnas de texto:\n')
for col in cat_cols:
    mask_blank = df[col].astype(str).str.strip() == ''
    n = mask_blank.sum()
    if n > 0:
        print(f'  ⚠️  [{col}]: {n} valores vacíos/espacios')
print('✅ Revisión completada.')

In [ ]:
# ─── Tipos de datos de columnas numéricas esperadas ──────────────────────────
numeric_expected = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
numeric_expected = [c for c in numeric_expected if c in df.columns]

print('📊 Tipos de datos de columnas numéricas esperadas:')
for col in numeric_expected:
    print(f'  [{col}]: dtype = {df[col].dtype} | Ejemplo: {df[col].iloc[0]!r}')

In [ ]:
# ─── Inconsistencias de capitalización / espacios ─────────────────────────────
print('🔡 Verificación de espacios iniciales/finales en columnas categóricas:\n')
for col in cat_cols:
    has_spaces = (df[col].astype(str) != df[col].astype(str).str.strip()).any()
    if has_spaces:
        print(f'  ⚠️  [{col}] tiene valores con espacios extra.')

# Verificar capitalización mixta usando pandas.unique()
print('\n🔠 Valores únicos que podrían tener inconsistencias de capitalización:')
for col in cat_cols:
    unique_vals = pd.unique(df[col].astype(str).str.lower())
    original_unique = pd.unique(df[col].astype(str))
    if len(original_unique) != len(unique_vals):
        print(f'  ⚠️  [{col}]: diferencia de {len(original_unique) - len(unique_vals)} categorías por capitalización')

print('\n✅ Verificación de inconsistencias completada.')

In [ ]:
# ─── Resumen visual de nulos ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
null_pct = (df.isnull().sum() / len(df) * 100)
null_pct = null_pct[null_pct > 0]

if null_pct.empty:
    ax.text(0.5, 0.5, '✅ Sin valores nulos detectados',
            ha='center', va='center', fontsize=14, color='green')
    ax.axis('off')
else:
    null_pct.sort_values(ascending=False).plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='white')
    ax.set_title('Porcentaje de Valores Nulos por Columna', fontweight='bold')
    ax.set_ylabel('% Nulos')
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

---
## 4️⃣ Limpieza y Tratamiento de Datos

Aplicamos las correcciones identificadas en el paso anterior:
- Normalización de strings (`.strip()`, `.lower()`, etc.)
- Conversión de tipos de datos
- Manejo de valores faltantes
- Estandarización de categorías


In [ ]:
# ─── Copia de trabajo ────────────────────────────────────────────────────────
df_clean = df.copy()
print(f'📋 Dimensiones originales: {df.shape}')

In [ ]:
# ─── 1. Limpiar espacios y estandarizar texto en columnas object ──────────────
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()            # eliminar espacios iniciales/finales
        .str.replace(r'\s+', ' ', regex=True)  # colapsar espacios múltiples
    )

print('✅ Espacios extra eliminados en todas las columnas de texto.')

In [ ]:
# ─── 2. Convertir TotalCharges a numérico (puede venir como string) ───────────
if 'TotalCharges' in df_clean.columns:
    # Reemplazar espacios vacíos por NaN y convertir
    df_clean['TotalCharges'] = (
        df_clean['TotalCharges']
        .replace(' ', np.nan)
        .replace('', np.nan)
    )
    df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
    n_nan_tc = df_clean['TotalCharges'].isna().sum()
    print(f'💰 TotalCharges convertido a float. NaN generados: {n_nan_tc}')

# ─── 3. Convertir MonthlyCharges a numérico ────────────────────────────────────
if 'MonthlyCharges' in df_clean.columns:
    df_clean['MonthlyCharges'] = pd.to_numeric(df_clean['MonthlyCharges'], errors='coerce')
    print('💰 MonthlyCharges convertido a float.')

# ─── 4. Convertir tenure a entero ─────────────────────────────────────────────
if 'tenure' in df_clean.columns:
    df_clean['tenure'] = pd.to_numeric(df_clean['tenure'], errors='coerce')
    print('📅 tenure convertido a numérico.')

In [ ]:
# ─── 5. Imputar NaN en TotalCharges con la mediana (clientes nuevos con tenure=0) ─
if 'TotalCharges' in df_clean.columns:
    mediana_tc = df_clean['TotalCharges'].median()
    before = df_clean['TotalCharges'].isna().sum()
    df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(mediana_tc)
    after = df_clean['TotalCharges'].isna().sum()
    print(f'🔧 TotalCharges: imputados {before - after} nulos con mediana (${mediana_tc:.2f})')

In [ ]:
# ─── 6. Estandarizar columna Churn ────────────────────────────────────────────
if 'Churn' in df_clean.columns:
    # Mapeamos posibles variaciones
    churn_map = {
        'yes': 'Yes', 'no': 'No',
        '1': 'Yes', '0': 'No',
        'true': 'Yes', 'false': 'No'
    }
    df_clean['Churn'] = (
        df_clean['Churn']
        .str.lower()
        .map(lambda x: churn_map.get(x, x.capitalize()))
    )
    print(f'✅ Columna Churn estandarizada. Valores únicos: {df_clean["Churn"].unique()}')

In [ ]:
# ─── 7. Crear columna numérica para Churn (0/1) ────────────────────────────────
df_clean['Churn_num'] = (df_clean['Churn'] == 'Yes').astype(int)
print('✅ Columna Churn_num creada (0 = No churn, 1 = Churn).')

In [ ]:
# ─── 8. Eliminar registros duplicados ────────────────────────────────────────
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
n_after = len(df_clean)
print(f'🗑️  Duplicados eliminados: {n_before - n_after}')
print(f'\n📋 Dimensiones finales del dataset limpio: {df_clean.shape}')

In [ ]:
# ─── Resumen final de calidad ─────────────────────────────────────────────────
print('📊 Resumen post-limpieza:\n')
print(f'  Filas       : {df_clean.shape[0]:,}')
print(f'  Columnas    : {df_clean.shape[1]}')
print(f'  Nulos totales: {df_clean.isnull().sum().sum()}')
print()
df_clean.dtypes.to_frame('dtype').T

---
## 5️⃣ Análisis Descriptivo Estadístico

Calculamos estadísticas descriptivas (media, mediana, desviación estándar, percentiles) tanto para variables numéricas como para variables categóricas.


In [ ]:
# ─── describe() para variables numéricas ─────────────────────────────────────
print('📈 Estadísticas descriptivas — Variables NUMÉRICAS:')
num_stats = df_clean.describe().T
num_stats['median'] = df_clean.describe().T.index.map(
    lambda c: df_clean[c].median() if c in df_clean.select_dtypes(include='number').columns else np.nan
)
display(num_stats)

In [ ]:
# ─── describe() para variables categóricas ────────────────────────────────────
print('🏷️  Estadísticas descriptivas — Variables CATEGÓRICAS:')
display(df_clean.describe(include='object'))

In [ ]:
# ─── Medidas adicionales ──────────────────────────────────────────────────────
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
num_cols = [c for c in num_cols if c in df_clean.columns]

print('📐 Métricas adicionales por variable numérica:\n')
stats_extra = pd.DataFrame(index=num_cols)
stats_extra['Media']     = [df_clean[c].mean()   for c in num_cols]
stats_extra['Mediana']   = [df_clean[c].median() for c in num_cols]
stats_extra['Moda']      = [df_clean[c].mode()[0] for c in num_cols]
stats_extra['Std']       = [df_clean[c].std()    for c in num_cols]
stats_extra['Varianza']  = [df_clean[c].var()    for c in num_cols]
stats_extra['Asimetría'] = [df_clean[c].skew()   for c in num_cols]
stats_extra['Curtosis']  = [df_clean[c].kurtosis() for c in num_cols]
stats_extra['Min']       = [df_clean[c].min()    for c in num_cols]
stats_extra['Max']       = [df_clean[c].max()    for c in num_cols]

display(stats_extra.round(2))

In [ ]:
# ─── Distribuciones de variables numéricas con histogramas ───────────────────
fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 5))
if len(num_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols):
    sns.histplot(df_clean[col], ax=ax, kde=True, color='#4C72B0', edgecolor='white', bins=30)
    ax.axvline(df_clean[col].mean(),   color='red',    linestyle='--', linewidth=1.5, label='Media')
    ax.axvline(df_clean[col].median(), color='orange', linestyle='--', linewidth=1.5, label='Mediana')
    ax.set_title(f'Distribución de {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de Variables Numéricas', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Boxplots para detectar outliers ─────────────────────────────────────────
fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 5))
if len(num_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols):
    sns.boxplot(y=df_clean[col], ax=ax, color='#a8d8ea', linewidth=1.5)
    ax.set_title(f'Boxplot — {col}', fontweight='bold')
    ax.set_ylabel(col)

plt.suptitle('Detección de Outliers (Boxplots)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Matriz de correlación ────────────────────────────────────────────────────
corr_cols = num_cols + ['Churn_num', 'SeniorCitizen'] if 'SeniorCitizen' in df_clean.columns else num_cols + ['Churn_num']
corr_cols = [c for c in corr_cols if c in df_clean.columns]

corr_matrix = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 11}
)
ax.set_title('Matriz de Correlación (Variables Numéricas)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6️⃣ Distribución del Churn (Evasión de Clientes)

Visualizamos la proporción de clientes que **cancelaron** vs los que **permanecieron**.


In [ ]:
# ─── Conteo y proporción ──────────────────────────────────────────────────────
churn_counts = df_clean['Churn'].value_counts()
churn_pct    = df_clean['Churn'].value_counts(normalize=True).mul(100).round(2)

print('📊 Distribución del Churn:')
summary_churn = pd.DataFrame({'Cantidad': churn_counts, 'Porcentaje (%)': churn_pct})
display(summary_churn)

In [ ]:
# ─── Pie chart + Bar chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#4C72B0', '#DD8452']

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    churn_counts,
    labels=churn_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    explode=[0, 0.05],
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(13)
    at.set_fontweight('bold')
axes[0].set_title('Proporción de Churn', fontweight='bold', fontsize=13)

# Bar chart
bars = axes[1].bar(
    churn_counts.index, churn_counts.values,
    color=colors, edgecolor='white', linewidth=1.5, width=0.5
)
for bar, val, pct in zip(bars, churn_counts.values, churn_pct.values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f'{val:,}\n({pct}%)',
        ha='center', va='bottom', fontweight='bold', fontsize=12
    )
axes[1].set_title('Conteo de Clientes por Churn', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Número de Clientes')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Distribución del Churn en Telecom X', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Gráfico interactivo con Plotly ──────────────────────────────────────────
fig_plotly = go.Figure(data=[
    go.Pie(
        labels=churn_counts.index,
        values=churn_counts.values,
        hole=0.4,
        marker=dict(colors=['#4C72B0', '#DD8452'],
                    line=dict(color='white', width=2)),
        textinfo='percent+label',
        hovertemplate='<b>%{label}</b><br>Clientes: %{value:,}<br>Porcentaje: %{percent}<extra></extra>'
    )
])
fig_plotly.update_layout(
    title=dict(text='🍩 Distribución del Churn — Donut Chart (Interactivo)', x=0.5, font_size=16),
    annotations=[dict(text='Churn', x=0.5, y=0.5, font_size=16, showarrow=False)],
    legend=dict(orientation='h', x=0.3, y=-0.1)
)
fig_plotly.show()

In [ ]:
# ─── Insight ──────────────────────────────────────────────────────────────────
churn_rate = churn_pct.get('Yes', churn_pct.get('yes', 0))
print(f'''
💡 INSIGHT:
   Aproximadamente el {churn_rate:.1f}% de los clientes han cancelado su servicio.
   Esto representa un desbalance de clases importante a tener en cuenta
   si se entrenaran modelos predictivos en etapas futuras.
''')

---
## 7️⃣ Churn por Variables Categóricas

Analizamos cómo se distribuye el churn según variables como **género, tipo de contrato, método de pago, servicios contratados**, etc.


In [ ]:
# ─── Función auxiliar para calcular tasa de churn ─────────────────────────────
def churn_rate_by(col, df=df_clean):
    """Devuelve un DataFrame con conteos y tasa de churn por categoría."""
    grp = df.groupby(col)['Churn_num'].agg(['sum', 'count'])
    grp.columns = ['Churn_Yes', 'Total']
    grp['Churn_Rate_%'] = (grp['Churn_Yes'] / grp['Total'] * 100).round(2)
    return grp.sort_values('Churn_Rate_%', ascending=False)

In [ ]:
# ─── Variables categóricas a analizar ─────────────────────────────────────────
cat_analysis = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'TechSupport', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]
cat_analysis = [c for c in cat_analysis if c in df_clean.columns]

print(f'📌 Columnas categóricas a analizar: {len(cat_analysis)}')
print(cat_analysis)

In [ ]:
# ─── Grid de barras: Tasa de Churn por variable categórica ───────────────────
n_cols = 3
n_rows = (len(cat_analysis) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4.5))
axes = axes.flatten()

for i, col in enumerate(cat_analysis):
    ax = axes[i]
    data = churn_rate_by(col).reset_index()

    bars = ax.bar(
        data[col].astype(str),
        data['Churn_Rate_%'],
        color=['#DD8452' if r > 20 else '#4C72B0' for r in data['Churn_Rate_%']],
        edgecolor='white', linewidth=1.2
    )
    for bar, rate in zip(bars, data['Churn_Rate_%']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{rate:.1f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold'
        )
    ax.set_title(f'Churn por {col}', fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Tasa de Churn (%)')
    ax.set_ylim(0, data['Churn_Rate_%'].max() * 1.2 + 5)
    ax.tick_params(axis='x', rotation=20)

# Ocultar ejes sobrantes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Tasa de Churn por Variables Categóricas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Gráfico interactivo: Churn por tipo de contrato ─────────────────────────
if 'Contract' in df_clean.columns:
    ct = churn_rate_by('Contract').reset_index()
    fig_ct = px.bar(
        ct, x='Contract', y='Churn_Rate_%',
        color='Churn_Rate_%',
        color_continuous_scale='RdYlGn_r',
        text='Churn_Rate_%',
        title='📄 Tasa de Churn por Tipo de Contrato (Interactivo)',
        labels={'Churn_Rate_%': 'Tasa de Churn (%)', 'Contract': 'Tipo de Contrato'}
    )
    fig_ct.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig_ct.update_layout(coloraxis_showscale=False, xaxis_title='Tipo de Contrato', height=450)
    fig_ct.show()

In [ ]:
# ─── Stacked bar: distribución de churn por método de pago ───────────────────
if 'PaymentMethod' in df_clean.columns:
    pm_ct = df_clean.groupby(['PaymentMethod', 'Churn']).size().unstack(fill_value=0)
    pm_ct_pct = pm_ct.div(pm_ct.sum(axis=1), axis=0) * 100

    ax = pm_ct_pct.plot(
        kind='barh', stacked=True, figsize=(12, 5),
        color=['#4C72B0', '#DD8452'], edgecolor='white'
    )
    ax.set_title('Distribución de Churn por Método de Pago (%)', fontweight='bold', fontsize=13)
    ax.set_xlabel('Porcentaje (%)')
    ax.set_ylabel('')
    ax.legend(title='Churn', bbox_to_anchor=(1, 1))
    ax.axvline(50, color='gray', linestyle='--', linewidth=1)
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Tabla resumen: Top variables por tasa de churn ──────────────────────────
print('🏆 TOP CATEGORÍAS con mayor tasa de Churn:\n')
all_churn_rows = []
for col in cat_analysis:
    data = churn_rate_by(col).reset_index()
    data['Variable'] = col
    data = data.rename(columns={col: 'Categoria'})
    all_churn_rows.append(data[['Variable', 'Categoria', 'Total', 'Churn_Yes', 'Churn_Rate_%']])

churn_summary = pd.concat(all_churn_rows).sort_values('Churn_Rate_%', ascending=False)
display(churn_summary.head(15))

---
## 8️⃣ Churn por Variables Numéricas

Analizamos cómo se distribuyen las variables numéricas (`tenure`, `MonthlyCharges`, `TotalCharges`) entre clientes que **cancelaron** y los que **no cancelaron**.


In [ ]:
# ─── Estadísticas por grupo de Churn ─────────────────────────────────────────
print('📊 Estadísticas de variables numéricas agrupadas por Churn:')
display(
    df_clean.groupby('Churn')[num_cols].agg(['mean', 'median', 'std']).round(2)
)

In [ ]:
# ─── KDE plots superpuestos por Churn ─────────────────────────────────────────
fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 6))
if len(num_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols):
    for churn_val, color in CHURN_PALETTE.items():
        subset = df_clean[df_clean['Churn'] == churn_val][col].dropna()
        sns.kdeplot(subset, ax=ax, color=color, fill=True, alpha=0.4, label=f'Churn={churn_val}')
    ax.set_title(f'Distribución de {col}\nsegun Churn', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Densidad')
    ax.legend(title='Churn')

plt.suptitle('Distribución de Variables Numéricas por Churn (KDE)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Boxplots por Churn ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 6))
if len(num_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols):
    sns.boxplot(
        data=df_clean, x='Churn', y=col, ax=ax,
        palette=CHURN_PALETTE, linewidth=1.5,
        order=['No', 'Yes']
    )
    # Overlay con puntos
    sns.stripplot(
        data=df_clean.sample(min(500, len(df_clean))),
        x='Churn', y=col, ax=ax,
        color='black', alpha=0.2, size=3, jitter=True,
        order=['No', 'Yes']
    )
    ax.set_title(f'{col} por Churn', fontweight='bold')
    ax.set_xlabel('Churn')
    ax.set_ylabel(col)

plt.suptitle('Boxplots de Variables Numéricas por Churn', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Violin plots con Plotly (interactivo) ────────────────────────────────────
fig_violin = make_subplots(rows=1, cols=len(num_cols),
                           subplot_titles=num_cols)

for i, col in enumerate(num_cols, 1):
    for churn_val, color in CHURN_PALETTE.items():
        subset = df_clean[df_clean['Churn'] == churn_val][col].dropna()
        fig_violin.add_trace(
            go.Violin(
                y=subset,
                name=f'Churn={churn_val}',
                box_visible=True,
                meanline_visible=True,
                fillcolor=color,
                opacity=0.7,
                line_color='rgba(0,0,0,0.5)',
                showlegend=(i == 1),
                legendgroup=churn_val
            ),
            row=1, col=i
        )

fig_violin.update_layout(
    title=dict(text='🎻 Violin Plots: Variables Numéricas por Churn (Interactivo)', x=0.5, font_size=15),
    violinmode='group',
    height=500,
    legend=dict(title='Churn')
)
fig_violin.show()

In [ ]:
# ─── Scatter: MonthlyCharges vs tenure coloreado por Churn ───────────────────
if 'MonthlyCharges' in df_clean.columns and 'tenure' in df_clean.columns:
    fig_scatter = px.scatter(
        df_clean,
        x='tenure',
        y='MonthlyCharges',
        color='Churn',
        color_discrete_map=CHURN_PALETTE,
        opacity=0.5,
        title='💸 MonthlyCharges vs Tenure — Coloreado por Churn (Interactivo)',
        labels={'tenure': 'Meses como Cliente (tenure)', 'MonthlyCharges': 'Cargo Mensual (USD)'},
        hover_data=['TotalCharges'] if 'TotalCharges' in df_clean.columns else None
    )
    fig_scatter.update_layout(height=500)
    fig_scatter.show()

In [ ]:
# ─── Histogramas apilados: tenure por Churn ───────────────────────────────────
if 'tenure' in df_clean.columns:
    fig_hist = px.histogram(
        df_clean, x='tenure', color='Churn',
        barmode='overlay',
        color_discrete_map=CHURN_PALETTE,
        nbins=50, opacity=0.7,
        title='📅 Distribución de Tenure por Churn (Interactivo)',
        labels={'tenure': 'Meses como Cliente', 'count': 'Cantidad'}
    )
    fig_hist.update_layout(bargap=0.1, height=450)
    fig_hist.show()

---
## 🏁 Conclusiones e Insights Principales

### Variables categóricas
- Los clientes con **contrato mes a mes** presentan una tasa de churn significativamente más alta que los de contratos anuales o bianuales.
- Los clientes con **facturación sin papel (PaperlessBilling = Yes)** tienden a cancelar más.
- El tipo de **servicio de internet** (especialmente Fiber Optic) está asociado a mayores tasas de churn.
- Clientes **sin pareja ni dependientes** tienen mayor propensión al churn.
- Los clientes que usan **Electronic Check** como método de pago muestran mayor churn que los que usan débito automático.

### Variables numéricas
- Clientes con **menor tenure** (antigüedad) tienden a cancelar más — los primeros meses son críticos para la retención.
- Los clientes que cancelan tienden a tener **MonthlyCharges más altos**.
- El **TotalCharges** es mayor en los clientes que permanecen (correlacionado con mayor antigüedad).

### Recomendaciones
1. **Incentivar contratos de largo plazo** para reducir churn.
2. **Foco en los primeros 6-12 meses** del cliente — implementar programas de onboarding y fidelización.
3. **Revisar precios de Fiber Optic** o mejorar la experiencia percibida.
4. Evaluar **modelos predictivos** (Random Forest, XGBoost) usando las variables identificadas como relevantes.

---
> *Análisis realizado como parte del Challenge Telecom X — Alura LATAM / ONE G9*
